# Download AIC 2026 Keyframes From Azure Blob

Notebook nay tai toan bo keyframe tu Azure Blob Storage ve Colab local SSD truoc:

`/content/aic_keyframes`

Sau khi tai xong, chay cell sync de copy sang Google Drive:

`/content/drive/MyDrive/AIC_2026/Keyframes`

Mac dinh notebook giu nguyen cau truc thu muc cua blob, vi du:

`L25/L25_V001/keyframe_0001.jpg`

## 1. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 2. Install Azure SDK

In [ ]:
!pip install -q azure-storage-blob tqdm

## 3. Configure Azure + output folder

Khuyen dung `AZURE_STORAGE_CONNECTION_STRING`. Neu khong co, co the dung `AZURE_STORAGE_ACCOUNT_NAME` + `AZURE_STORAGE_PRIMARY_KEY`.

Notebook se uu tien doc tu Colab Secrets neu ban tao cac secret cung ten; neu khong co thi se hoi nhap thu cong.

In [ ]:
import getpass
import os
import shutil
from pathlib import Path

def read_secret(name, *, required=False):
    value = os.environ.get(name, '').strip()
    if value:
        return value
    try:
        from google.colab import userdata
        value = (userdata.get(name) or '').strip()
        if value:
            return value
    except Exception:
        pass
    if required:
        return getpass.getpass(f'{name}: ').strip()
    return ''

# Azure settings
AZURE_STORAGE_CONNECTION_STRING = read_secret('AZURE_STORAGE_CONNECTION_STRING')
AZURE_STORAGE_ACCOUNT_NAME = read_secret('AZURE_STORAGE_ACCOUNT_NAME')
AZURE_STORAGE_PRIMARY_KEY = read_secret('AZURE_STORAGE_PRIMARY_KEY')

# Neu keyframes cua ban nam o container/prefix khac, sua 2 dong nay.
KEYFRAME_CONTAINER = os.environ.get('AZURE_BLOB_CONTAINER_KEYFRAMES', 'keyframes').strip() or 'keyframes'
KEYFRAME_PREFIX = os.environ.get('AZURE_KEYFRAME_PREFIX', '').strip().strip('/')

# Fast path: download many small files to Colab local SSD first.
# Writing hundreds of thousands of files directly to Google Drive mount is much slower.
OUTPUT_DIR = Path('/content/aic_keyframes')
DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/AIC_2026/Keyframes')

# Resume mode: skip file local neu size trung voi blob.
OVERWRITE = False
MAX_WORKERS = 32
HTTP_POOL_SIZE = 128
PER_BLOB_MAX_CONCURRENCY = 1

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Container:', KEYFRAME_CONTAINER)
print('Prefix:', KEYFRAME_PREFIX or '<root>')
print('Output:', OUTPUT_DIR)
print('To copy into Drive after download: run the sync cell below.')
print('Drive sync target:', DRIVE_OUTPUT_DIR)

## 4. Connect and list keyframe blobs

In [ ]:
from azure.core.pipeline.transport import RequestsTransport
from azure.storage.blob import BlobServiceClient

transport = RequestsTransport(connection_pool_maxsize=HTTP_POOL_SIZE)
if AZURE_STORAGE_CONNECTION_STRING:
    service = BlobServiceClient.from_connection_string(AZURE_STORAGE_CONNECTION_STRING, transport=transport)
elif AZURE_STORAGE_ACCOUNT_NAME and AZURE_STORAGE_PRIMARY_KEY:
    service = BlobServiceClient(
        account_url=f'https://{AZURE_STORAGE_ACCOUNT_NAME}.blob.core.windows.net',
        credential=AZURE_STORAGE_PRIMARY_KEY,
        transport=transport,
    )
else:
    AZURE_STORAGE_CONNECTION_STRING = read_secret('AZURE_STORAGE_CONNECTION_STRING', required=True)
    service = BlobServiceClient.from_connection_string(AZURE_STORAGE_CONNECTION_STRING, transport=transport)

container = service.get_container_client(KEYFRAME_CONTAINER)

list_prefix = f'{KEYFRAME_PREFIX}/' if KEYFRAME_PREFIX else None
blobs = [b for b in container.list_blobs(name_starts_with=list_prefix)]
blobs = [b for b in blobs if not b.name.endswith('/')]

total_bytes = sum(int(b.size or 0) for b in blobs)
print(f'Found {len(blobs):,} blobs')
print(f'Total size: {total_bytes / (1024 ** 3):.2f} GiB')
free_bytes = shutil.disk_usage(OUTPUT_DIR.anchor or '/content').free
print(f'Free space near output: {free_bytes / (1024 ** 3):.2f} GiB')
if OUTPUT_DIR.as_posix().startswith('/content/') and total_bytes > free_bytes * 0.9:
    print('WARNING: /content may not have enough space. Either narrow KEYFRAME_PREFIX or set OUTPUT_DIR to Drive.')
print('First 5 blobs:')
for b in blobs[:5]:
    print(' -', b.name, f'({int(b.size or 0):,} bytes)')

## 5. Download all keyframes

Cell nay co the chay lai. File nao da tai du size se duoc skip.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

def local_path_for_blob(blob_name):
    relative = blob_name
    if KEYFRAME_PREFIX and relative.startswith(KEYFRAME_PREFIX + '/'):
        relative = relative[len(KEYFRAME_PREFIX) + 1:]
    relative = relative.lstrip('/').replace('\\\\', '/')
    return OUTPUT_DIR / relative

def should_skip(path, size):
    if OVERWRITE or not path.exists():
        return False
    try:
        return path.stat().st_size == int(size or 0)
    except OSError:
        return False

def download_one(blob_item):
    blob_name = blob_item.name
    expected_size = int(blob_item.size or 0)
    out_path = local_path_for_blob(blob_name)
    if should_skip(out_path, expected_size):
        return ('skipped', blob_name, expected_size, '')

    out_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = out_path.with_name(out_path.name + '.tmp')
    blob_client = container.get_blob_client(blob_name)

    try:
        with tmp_path.open('wb') as f:
            stream = blob_client.download_blob(max_concurrency=PER_BLOB_MAX_CONCURRENCY)
            stream.readinto(f)
        actual_size = tmp_path.stat().st_size
        if expected_size and actual_size != expected_size:
            tmp_path.unlink(missing_ok=True)
            return ('failed', blob_name, expected_size, f'size mismatch: {actual_size} != {expected_size}')
        tmp_path.replace(out_path)
        return ('downloaded', blob_name, expected_size, '')
    except Exception as exc:
        try:
            tmp_path.unlink(missing_ok=True)
        except Exception:
            pass
        return ('failed', blob_name, expected_size, repr(exc))

started = time.time()
counts = {'downloaded': 0, 'skipped': 0, 'failed': 0}
failed = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(download_one, b) for b in blobs]
    with tqdm(total=len(futures), unit='file') as pbar:
        for future in as_completed(futures):
            status, blob_name, size, error = future.result()
            counts[status] = counts.get(status, 0) + 1
            if status == 'failed':
                failed.append((blob_name, error))
            pbar.set_postfix(counts)
            pbar.update(1)

elapsed = time.time() - started
print('Done')
print(counts)
print(f'Elapsed: {elapsed / 60:.1f} minutes')
print('Output:', OUTPUT_DIR)
print('To copy into Drive after download: run the sync cell below.')

if failed:
    print('\nFailed samples:')
    for name, error in failed[:20]:
        print('-', name, error)
    print(f'Total failed: {len(failed):,}')

## 6. Sync to Google Drive

Only run this after the download cell finishes. Local SSD download is faster; this cell copies the completed tree into Drive with resume support.

In [ ]:
import subprocess

DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cmd = ['rsync', '-a', '--info=progress2', f'{OUTPUT_DIR}/', f'{DRIVE_OUTPUT_DIR}/']
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)
print('Drive output:', DRIVE_OUTPUT_DIR)

## 7. Quick verify local files

In [ ]:
local_files = [p for p in OUTPUT_DIR.rglob('*') if p.is_file() and not p.name.endswith('.tmp')]
local_bytes = sum(p.stat().st_size for p in local_files)

print(f'Local files: {len(local_files):,}')
print(f'Local size: {local_bytes / (1024 ** 3):.2f} GiB')
print('Output:', OUTPUT_DIR)
print('To copy into Drive after download: run the sync cell below.')
print('Sample files:')
for p in local_files[:10]:
    print('-', p.relative_to(OUTPUT_DIR))